In [107]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import importlib

from sklearn.metrics import accuracy_score, precision_score, recall_score
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow.keras import layers, losses
from tensorflow.keras.models import Model

import source.anomalyDetector as ad
importlib.reload(ad)

import source.data_wrangling as dw
importlib.reload(dw)

<module 'source.data_wrangling' from '/home/jotje3041/thesis_2_electric_boogaloo/fystek-assignments/source/data_wrangling.py'>

## Autoencoder using differently structured data (not complete)

#### Reading the data

In [108]:
ds_2025 = xr.open_dataset("files/2025_KVS_deployment_flagged.nc")
ds_2024 = xr.open_dataset("files/2024_KVS_deployment_flagged.nc")

In [109]:
ds_2025_1m_temp = ds_2025["temp_1m_calibrated"]
ds_2025_ir_temp = ds_2025["temp_snowsurface"]
ds_2025["temp_diff"] = ds_2025_1m_temp - ds_2025_ir_temp

ds_2024_1m_temp = ds_2024["temp_1m_calibrated"]
ds_2024_ir_temp = ds_2024["temp_snowsurface_calibrated"]
ds_2024["temp_diff"] = ds_2024_1m_temp - ds_2024_ir_temp

In [110]:
dataset_2025 = ds_2025[["temp_1m_calibrated","temp_diff"]]
labels_2025 = ds_2025["temp_1m_quality_flag"]

dataset_2024 = ds_2024[["temp_1m_calibrated","temp_diff"]]
labels_2024 = ds_2024["temp_1m_quality_flag"]

#### Scaling the data

In [111]:
def scale_dataset(dataset):
    datasets = []
    for traj in range(len(dataset.trajectory.values)):
        sel_data = dataset.isel(trajectory=traj)

        min_val = np.nanmin(sel_data["temp_1m_calibrated"])
        max_val = np.nanmax(sel_data["temp_1m_calibrated"])

        sel_data["temp_1m_calibrated"] =  (sel_data["temp_1m_calibrated"] - min_val) / (max_val - min_val)
        
        min_val = np.nanmin(sel_data["temp_diff"])
        max_val = np.nanmax(sel_data["temp_diff"])

        sel_data["temp_diff"] =  (sel_data["temp_diff"] - min_val) / (max_val - min_val)
        
        datasets.append(sel_data)    

    dataset_scaled = xr.concat(datasets,dim="trajectory")

    return dataset_scaled

In [112]:
dataset_2025_scaled = scale_dataset(dataset_2025)
#for traj in range(len(dataset_2025_scaled.trajectory)): dataset_2025_scaled["temp_1m_calibrated"].isel(trajectory=traj).plot()

In [113]:
dataset_2024_scaled = scale_dataset(dataset_2024)
#for traj in range(len(dataset_2024_scaled.trajectory)): dataset_2024_scaled["temp_1m_calibrated"].isel(trajectory=traj).plot()

#### Reshape the data

We standardize the input dataset, making sure that the measurement sample dimension is the same for both datasets (to make it easier for the autoencoder) padding with Nan values where there are no measurements. (This nan padding technique already exists in the dataset to account for variable buoy life span).

The label array is padded with zeroes, since that is coherent with the way that the padding is handled in the dataset.

The goal is to achieve a data array that is number of bouys x 2 x maximum number of measurements, so that it is like a stack of 2 x num measurements frames so that the autoencoder only has to keep track of the two constant 2 x num measurement dimensions.

In [114]:
max_measurement_len = np.max([ds_2025["time_temp"].sizes["obs_temp"],ds_2024["time_temp"].sizes["obs_temp"]])

nr_bouys_2025 = len(ds_2025.trajectory)
nr_bouys_2024 = len(ds_2024.trajectory)

array_data_2025 = np.full((nr_bouys_2025,2,max_measurement_len),np.nan)
array_labels_2025 = np.full((nr_bouys_2025,2,max_measurement_len),0)

array_data_2024 = np.full((nr_bouys_2024,2,max_measurement_len),np.nan)
array_labels_2024 = np.full((nr_bouys_2024,2,max_measurement_len),0)

print(f"2025 data shape: {array_data_2025.shape}, 2024 data shape: {array_data_2024.shape}")
print(f"2025 label shape: {array_labels_2025.shape}, 2024 label shape:{array_labels_2024.shape}")

2025 data shape: (20, 2, 3145), 2024 data shape: (33, 2, 3145)
2025 label shape: (20, 2, 3145), 2024 label shape:(33, 2, 3145)


In [115]:
array_data_2025 = dw.fill_data(dataset_2025,array_data_2025,["temp_1m_calibrated","temp_diff"])
array_data_2024 = dw.fill_data(dataset_2024,array_data_2024,["temp_1m_calibrated","temp_diff"])

In [116]:
array_labels_2025 = dw.fill_labels(labels_2025,array_labels_2025)
array_labels_2024 = dw.fill_labels(labels_2024,array_labels_2024)

#### Prepare train/test data from the 2025 dataset

In [117]:
train_data,test_data,train_labels,test_labels = train_test_split(array_data_2025,
                                                                 array_labels_2025,
                                                                 test_size=0.2)



In [118]:
train_data.shape,test_data.shape,train_labels.shape

((16, 2, 3145), (4, 2, 3145), (16, 2, 3145))

In [104]:
normal_train_data = np.empty(train_data.shape)
normal_test_data = np.empty(test_data.shape)

anomalous_train_data = np.empty(train_data.shape)
anomalous_test_data = np.empty(test_data.shape)

In [ ]:
train_labels = train_labels.astype(bool)
test_labels = test_labels.astype(bool)

normal_train_data = train_data[~train_labels]
print(normal_train_data.shape)

normal_test_data = test_data[~test_labels]
normal_test_data = test_data[:,1,:][~test_labels]

anomalous_train_data = train_data[:,0,:][train_labels]
anomalous_train_data = train_data[:,1,:][train_labels]

anomalous_test_data = test_data[:,0,:][test_labels]
anomalous_test_data = test_data[:,1,:][test_labels]

#### Train autoencoder

In [99]:
importlib.reload(ad)

autoencoder = ad.AnomalyDetector()
autoencoder.compile(optimizer='adam',loss='mae')

In [102]:
normal_train_data.shape

(47634,)

In [ ]:
history = autoencoder.fit(normal_train_data, normal_train_data,
          epochs=20,
          batch_size=512,
          validation_data=(test_data, test_data),
          shuffle=True)

In [ ]:
plt.plot(history.history["loss"], label="Training Loss")
plt.plot(history.history["val_loss"], label="Validation Loss")
plt.legend()


In [ ]:
encoded_data = autoencoder.encoder(normal_test_data).numpy()
decoded_data = autoencoder.decoder(encoded_data).numpy()

plt.figure(figsize=(20,10))

ax = plt.subplot(2,1,1)

ax.plot(normal_test_data["temp_diff"].values, 'b')
ax.plot(decoded_data[:,0], 'r')

ax.set_title("Normal test data input vs. reconstruction,\n temperature difference")
ax.legend(labels=["Input", "Reconstruction"])

ax1 = plt.subplot(2,1,2)

ax1.plot(normal_test_data["temp_1m_calibrated"].values, 'b')
ax1.plot(decoded_data[:,1], 'r')

ax1.set_title("Normal test data input vs. reconstruction,\n 1-m temperature")
ax1.legend(labels=["Input", "Reconstruction"])



In [ ]:
encoded_data = autoencoder.encoder(anomalous_test_data).numpy()
decoded_data = autoencoder.decoder(encoded_data).numpy()


plt.figure(figsize=(20,10))

ax = plt.subplot(2,1,1)

ax.plot(anomalous_test_data["temp_diff"].values, 'b')
ax.plot(decoded_data[:,0], 'r')

ax.set_title("Anomalous test data input vs. reconstruction,\n temperature difference")
ax.legend(labels=["Input", "Reconstruction"])

ax1 = plt.subplot(2,1,2)

ax1.plot(anomalous_test_data["temp_1m_calibrated"].values, 'b')
ax1.plot(decoded_data[:,1], 'r')

ax1.set_title("Anomalous test data input vs. reconstruction,\n 1-m temperature")
ax1.legend(labels=["Input", "Reconstruction"])


Detect anomalies by calculating whether the reconstruction loss is greater than a fixed threshold. 

In [ ]:
#=======================================================================
#Normal train
reconstructions = autoencoder.predict(normal_train_data)
train_loss = tf.keras.losses.mae(reconstructions, normal_train_data)

fig = plt.figure(figsize=(8,10))

ax = plt.subplot(2,1,1)

ax.hist(train_loss, bins=50)
ax.set_xlabel("Train loss")
ax.set_ylabel("No of examples")
ax.set_title("Training loss\n Normal training data")

threshold = np.mean(train_loss) + np.std(train_loss)
ax.text(0.15, 6000, f'Threshold: {threshold}', fontsize=12)

#=======================================================================
#Anomalous train

reconstructions = autoencoder.predict(anomalous_test_data)
test_loss = tf.keras.losses.mae(reconstructions, anomalous_test_data)

ax1 = plt.subplot(2,1,2)
ax1.hist(test_loss, bins=50)
ax1.set_xlabel("Test loss")
ax1.set_ylabel("No of examples")
ax1.set_title("Training loss\n Anomalous test data")
ax1.axvline(x=threshold,color='r',label="threshold")
ax1.legend()

plt.tight_layout()


In [ ]:
preds = ad.anomalyPredict(autoencoder, test_data, threshold)
ad.print_stats(preds, test_labels)

### Using the 2025 dataset to flag the 2024 dataset

In [ ]:
preds = ad.anomalyPredict(autoencoder, dataset_2024_scaled, threshold)
ad.print_stats(preds, test_labels)